# The Phillips Curve in Brazil
### An automated empirical analysis of the inflation–unemployment tradeoff

**Author:** Tiago Ferreira Rodrigues — B.Sc. Economics, FGV EPGE

---

**Research question.** Does Brazil exhibit the inverse inflation–unemployment relationship predicted by the Phillips Curve? How stable is that relationship across business-cycle regimes?

**Approach.**
1. **Automated data acquisition** — both series are pulled directly from the Central Bank of Brazil's SGS API at runtime (no manual downloads), with local caching for reproducibility and offline use.
2. **Validation** — programmatic coherence checks (range, duplicates, missing values) before any analysis.
3. **Estimation** — OLS with HAC (Newey–West) standard errors, contemporaneous and lagged specifications, and a rolling-correlation diagnostic of structural stability.

**Data.**

| Series | SGS code | Frequency | Coverage |
|---|---|---|---|
| IPCA — monthly variation (%) | 433 | Monthly | 1980– |
| Unemployment rate — PNAD Contínua (%) | 24369 | Monthly (moving quarter) | 2012– |

The effective sample is **2012 onward**, constrained by PNADc availability. Monthly IPCA variations are compounded into **12-month (YoY) inflation**, the measure typically used in Phillips Curve estimations, which also smooths month-to-month noise.

In [ ]:
# ── Imports & configuration ──────────────────────────────────────────
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

CACHE_DIR = Path("data")
CACHE_DIR.mkdir(exist_ok=True)

START_DATE = "01/01/2012"   # PNADc starts in 2012

## 1. Automated data acquisition — BCB SGS API

Instead of relying on manually downloaded CSVs, both series are fetched directly from the [SGS API](https://dadosabertos.bcb.gov.br/) maintained by the Central Bank of Brazil. Each call is cached to `data/` so the notebook:

- always reflects the **latest official data** when online;
- remains **fully reproducible offline** (falls back to the cache);
- requires **zero manual preprocessing** — no encoding fixes, no footer rows to strip.

In [ ]:
def fetch_sgs(series_code: int, name: str, start: str = START_DATE) -> pd.Series:
    """Fetch a time series from the BCB SGS API, with local CSV caching.

    Parameters
    ----------
    series_code : SGS series identifier (e.g. 433 for monthly IPCA).
    name        : column name for the returned series.
    start       : start date in dd/mm/yyyy format.

    Returns
    -------
    pd.Series indexed by month-start dates.
    """
    cache_file = CACHE_DIR / f"sgs_{series_code}.csv"
    url = (
        f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{series_code}/dados"
        f"?formato=json&dataInicial={start}"
    )
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        df = pd.DataFrame(resp.json())
        df.to_csv(cache_file, index=False)
        print(f"[online]  SGS {series_code} → {len(df)} observations (cache updated)")
    except (requests.RequestException, ValueError) as err:
        if not cache_file.exists():
            raise RuntimeError(
                f"SGS {series_code}: API unavailable and no local cache found."
            ) from err
        df = pd.read_csv(cache_file)
        print(f"[cached]  SGS {series_code} → {len(df)} observations ({err.__class__.__name__})")

    s = (
        df.assign(
            date=pd.to_datetime(df["data"], format="%d/%m/%Y"),
            value=pd.to_numeric(df["valor"], errors="coerce"),
        )
        .set_index("date")["value"]
        .rename(name)
        .sort_index()
    )
    return s


ipca_monthly = fetch_sgs(433, "ipca_monthly")        # IPCA, monthly % variation
unemployment = fetch_sgs(24369, "unemployment")      # PNADc unemployment rate, %

## 2. Preprocessing & validation

Monthly IPCA variations are compounded into a 12-month accumulated rate:

$$\pi_t^{12m} = \left[\prod_{j=0}^{11}\left(1 + \frac{\pi_{t-j}}{100}\right) - 1\right] \times 100$$

The two series are then merged on the monthly index. Before analysis, the merged panel passes through programmatic coherence checks — failing any of them halts the notebook rather than silently producing misleading results.

In [ ]:
# ── 12-month accumulated inflation ───────────────────────────────────
inflation_yoy = (
    ((1 + ipca_monthly / 100).rolling(12).apply(np.prod, raw=True) - 1) * 100
).rename("inflation_yoy")

df = pd.concat([inflation_yoy, unemployment], axis=1).dropna()

print(f"Sample: {df.index.min():%Y-%m} → {df.index.max():%Y-%m}  ({len(df)} monthly obs.)")
df.describe().round(2)

In [ ]:
# ── Validation: fail fast on incoherent data ─────────────────────────
checks = {
    "Unemployment within [0, 100]": df["unemployment"].between(0, 100).all(),
    "Inflation within plausible range (-5, 25)": df["inflation_yoy"].between(-5, 25).all(),
    "No duplicate dates": not df.index.duplicated().any(),
    "No missing values": not df.isna().any().any(),
    "Monotonically increasing index": df.index.is_monotonic_increasing,
}

for check, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {check}")

assert all(checks.values()), "Data validation failed — inspect before proceeding."

## 3. Exploratory analysis

The two series on separate axes (their scales differ): visually, the post-2015 recession pushes **both** inflation and unemployment up — the first hint that a single, stable Phillips relationship may not hold across the full sample.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5.5))

ax1.plot(df.index, df["inflation_yoy"], color="#1f4e79", lw=2, label="Inflation (IPCA, 12m %)")
ax1.set_ylabel("Inflation — IPCA 12m (%)", color="#1f4e79")
ax1.tick_params(axis="y", labelcolor="#1f4e79")

ax2 = ax1.twinx()
ax2.plot(df.index, df["unemployment"], color="#c0504d", lw=2, label="Unemployment (PNADc, %)")
ax2.set_ylabel("Unemployment rate (%)", color="#c0504d")
ax2.tick_params(axis="y", labelcolor="#c0504d")
ax2.grid(False)

# Shade recession / pandemic episodes
for span, lbl in [(("2015-01", "2016-12"), "2015–16 recession"),
                  (("2020-03", "2021-12"), "Pandemic")]:
    ax1.axvspan(pd.Timestamp(span[0]), pd.Timestamp(span[1]), color="grey", alpha=0.15)

ax1.xaxis.set_major_locator(mdates.YearLocator(2))
ax1.set_title("Inflation and unemployment in Brazil (2013–present)", fontsize=13)
fig.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

## 4. The Phillips Curve, by business-cycle regime

A single pooled scatter hides the most interesting feature of the Brazilian data: the inflation–unemployment relationship **shifts across regimes**. Coloring observations by macroeconomic episode makes the instability visible — points from the 2015–16 recession sit in the high-inflation/high-unemployment quadrant, a configuration the textbook Phillips Curve rules out (and which the literature attributes to supply shocks and de-anchored expectations rather than demand dynamics).

In [ ]:
# ── Business-cycle regimes ───────────────────────────────────────────
def assign_regime(d: pd.Timestamp) -> str:
    if d.year <= 2014:
        return "2013–2014 · Commodity boom tail"
    if d.year <= 2016:
        return "2015–2016 · Recession"
    if d.year <= 2019:
        return "2017–2019 · Slow recovery"
    if d.year <= 2021:
        return "2020–2021 · Pandemic"
    return "2022– · Post-pandemic"

df["regime"] = [assign_regime(d) for d in df.index]

fig, ax = plt.subplots(figsize=(10.5, 6.5))
sns.scatterplot(
    data=df, x="unemployment", y="inflation_yoy",
    hue="regime", palette="viridis", s=55, alpha=0.85, ax=ax,
)

# Pooled OLS fit line for reference
b, a = np.polyfit(df["unemployment"], df["inflation_yoy"], 1)
xs = np.linspace(df["unemployment"].min(), df["unemployment"].max(), 100)
ax.plot(xs, a + b * xs, color="black", ls="--", lw=1.5, label="Pooled OLS fit")

ax.set_xlabel("Unemployment rate (%)")
ax.set_ylabel("Inflation — IPCA 12m (%)")
ax.set_title("Phillips Curve in Brazil — observations colored by regime", fontsize=13)
ax.legend(title=None, fontsize=9)
plt.tight_layout()
plt.show()

### 4.1 Estimation

The contemporaneous specification is

$$\pi_t = \alpha + \beta\, u_t + \varepsilon_t$$

Standard errors are **HAC (Newey–West)** with 12 lags — both series are strongly autocorrelated, so conventional OLS standard errors would dramatically overstate precision. The estimates below are **descriptive correlations, not causal slopes**: unemployment is endogenous to monetary policy and supply shocks, and a structural Phillips Curve would require controlling for inflation expectations (e.g., Focus survey) and imported-inflation shocks.

In [ ]:
X = sm.add_constant(df["unemployment"])
ols = sm.OLS(df["inflation_yoy"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 12})

r, p = stats.pearsonr(df["unemployment"], df["inflation_yoy"])
print(f"Pearson correlation: r = {r:.3f} (p = {p:.4f})\n")
print(ols.summary())

## 5. Lag structure

In New Keynesian Phillips Curve (NKPC) formulations, inflation responds to economic slack with a delay — pricing decisions are staggered (Calvo) and expectations adjust gradually. If the Brazilian relationship operates through that channel, **lagged** unemployment should correlate more strongly with current inflation than contemporaneous unemployment does.

In [ ]:
lag_results = {}
for lag in [0, 3, 6, 12]:
    d = df.assign(u_lag=df["unemployment"].shift(lag)).dropna()
    m = sm.OLS(d["inflation_yoy"], sm.add_constant(d["u_lag"])).fit(
        cov_type="HAC", cov_kwds={"maxlags": 12}
    )
    lag_results[f"u(t−{lag})"] = {
        "β (slope)": m.params["u_lag"],
        "HAC p-value": m.pvalues["u_lag"],
        "R²": m.rsquared,
        "N": int(m.nobs),
    }

pd.DataFrame(lag_results).T.round(3)

## 6. Is the relationship stable? Rolling correlation

A 24-month rolling Pearson correlation between inflation and unemployment shows **when** the Phillips relationship holds and when it breaks down. Sustained excursions into positive territory — periods where inflation and unemployment rise *together* — flag supply-driven episodes incompatible with a demand-side Phillips mechanism.

In [ ]:
roll_corr = df["inflation_yoy"].rolling(24).corr(df["unemployment"])

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(roll_corr.index, roll_corr, color="#1f4e79", lw=2)
ax.axhline(0, color="black", lw=0.8)
ax.fill_between(roll_corr.index, roll_corr, 0,
                where=(roll_corr > 0), color="#c0504d", alpha=0.25,
                label="ρ > 0 (anti-Phillips episodes)")
ax.fill_between(roll_corr.index, roll_corr, 0,
                where=(roll_corr <= 0), color="#1f4e79", alpha=0.15,
                label="ρ < 0 (Phillips-consistent)")
ax.set_title("24-month rolling correlation: inflation × unemployment", fontsize=13)
ax.set_ylabel("Pearson ρ (24m window)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Conclusions

1. **A negative inflation–unemployment correlation exists in the pooled sample, but it is weak and unstable.** The pooled slope is negative, yet its magnitude and significance depend heavily on the window examined.

2. **Regime heterogeneity is the dominant feature of the data.** The 2015–16 recession and the post-pandemic period place observations in the high-inflation/high-unemployment quadrant — consistent with supply shocks and expectation de-anchoring, not with movements along a stable Phillips Curve.

3. **The lag structure is informative.** Comparing contemporaneous and lagged specifications speaks to the sluggish price adjustment embedded in NKPC-style models, where staggered pricing delays the pass-through from slack to inflation.

4. **These are descriptive estimates.** A structural estimate would require, at minimum: inflation expectations (BCB Focus survey, SGS available), an imported-inflation control (exchange rate / commodity prices), and an identification strategy for the endogeneity of unemployment.

### Possible extensions
- Replace headline IPCA with **services core inflation** (SGS 10844), the component most directly linked to domestic slack.
- Add **Focus survey expectations** to estimate an expectations-augmented Phillips Curve.
- Estimate the **unemployment gap** (deviation from a trend-based NAIRU proxy, e.g. HP filter) instead of the unemployment level.